In [3]:
from src.mnist_model import MNIST_CNN
from src.training import get_accuracy
from src.utils import device
from settings import settings
from torch.utils.data import DataLoader
import torch

model = MNIST_CNN()
model.to(device)
model.load_state_dict(torch.load(settings.models_path / "mnist_model.pth"))
from mnist_training import test_data, test_labels, train_data
model.binary_mode()
print(get_accuracy(model, test_data, test_labels))
from src.improved_model import binary_sign

model.binary_mode()
layer1 = model.flatten(model.layer1.activation(model.layer1(train_data))).detach()
layer2 = model.layer3.activation(model.layer3(layer1)).detach()
layer3 = model.layer4.activation(model.layer4(layer2)).detach()


layer1 = layer1.cpu()
layer2 = layer2.cpu()
layer3 = layer3.cpu()


C:\Users\frrit\AppData\Local\Temp\ipykernel_8680\589547341.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(settings.models_path / "mnis

0.9521


TypeError: can't convert cuda:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.

In [9]:
from typing import Optional
from functools import lru_cache
import numpy as np
from scipy.optimize import minimize_scalar


x = layer1.numpy() == 1
y = layer2[:, 53].numpy() == 1

x_negative = x[~y, :]
x_positive = x[y, :]

A = np.random.rand(x.shape[1]) > 1
#A = np.zeros(x.shape[1]) == 1

def get_accuracy(z_negative: np.ndarray, z_positive: np.ndarray, threshold: Optional[int] = None) -> [float, int]:
    def outer_accuracy(threshold):
        threshold = int(threshold)
        return inner_accuracy(threshold)
    @lru_cache(None)
    def inner_accuracy(threshold):
        correct_positive = sum(z_positive >= threshold)
        correct_negative = sum(z_negative < threshold)
        return -(correct_negative + correct_positive)  # Negative for maximization

    if threshold is None:
        min_t, max_t = np.min([np.min(z_negative), np.min(z_positive)]), np.max([np.max(z_negative), np.max(z_positive)])
        bracket = (min_t - 1, max_t + 1)
    else:
        bracket = (threshold - 5, threshold + 5)

    result = minimize_scalar(outer_accuracy, bracket=bracket, method='golden')
    best_threshold = int(result.x)
    best_accuracy = -result.fun / (len(z_negative) + len(z_positive))

    return best_accuracy, best_threshold

z_negative = np.sum(~(np.bitwise_xor(x_negative, A)), axis=1)
z_positive = np.sum(~(np.bitwise_xor(x_positive, A)), axis=1)
best_accuracy, best_threshold = get_accuracy(z_negative, z_positive)

for epoch in range(10):
    best_before_epoch = best_accuracy
    for i in np.random.permutation(len(A)):
        # Subtract the old contribution
        z_negative -= (x_negative[:, i] == A[i])
        z_positive -= (x_positive[:, i] == A[i])

        # Flip the bit in A
        A[i] = ~A[i]

        # Add the new contribution
        z_negative += (x_negative[:, i] == A[i])
        z_positive += (x_positive[:, i] == A[i])

        # Compute accuracy and threshold
        accuracy, threshold = get_accuracy(z_negative, z_positive, best_threshold)

        # Update best accuracy and threshold
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_threshold = threshold
        else:
            # Revert the flip if accuracy doesn't improve
            z_negative -= (x_negative[:, i] == A[i])
            z_positive -= (x_positive[:, i] == A[i])
            A[i] = ~A[i]
            z_negative += (x_negative[:, i] == A[i])
            z_positive += (x_positive[:, i] == A[i])

        print(best_accuracy)

    if best_accuracy == best_before_epoch:
        print("Stalled")
        break


In [10]:
np.bincount(y)

Computing row 1
